# Watchlist example notebook

```Author: Francisco Förster, modifications: Kay Medina, Alejandra Muñoz Arancibia. Last updated: 20260626```

Here we will do a xmatch against a list of positions in the sky, given in the file ```watchlist.csv```, using the ALeRCE Table Access Protocol (TAP) service.

*It is highly recommended that you try this notebook in Google Colab using the following [link](https://colab.research.google.com/github/alercebroker/usecases/blob/master/notebooks/ZTF/ALeRCE_Other_Watchlist.ipynb).*
This will avoid you from having to sort out library installation problems and focus on the contents of the tutorial. You can try installing the dependencies later in your own system.

In [1]:
import pandas as pd

In [2]:
#!pip install pyvo
import pyvo

### TAP connection setup

Instead of using a direct PostgreSQL connection, we connect to the ALeRCE ZTF legacy database using Astronomical Data Query Language ([ADQL](https://www.ivoa.net/documents/ADQL/)) queries to the ALeRCE TAP service, returning results as pandas dataframes. Database table names are prefixed with `ztf.` (e.g. `ztf.object`, `ztf.detection`, etc.).

In [3]:
TAP_URL = "https://tap.alerce.online/tap"
tap_service = pyvo.dal.TAPService(TAP_URL)

def tap_query(query, maxrec=10_000_000):
    result = tap_service.search(query, maxrec=maxrec)
    return result.to_table().to_pandas()

def cone_search_per_position(df, build_query):
    frames = [tap_query(build_query(row)) for _, row in df.iterrows()]
    return pd.concat(frames, ignore_index=True)

# Get watchlist of object positions

In [4]:
df = pd.read_csv("https://github.com/alercebroker/usecases/blob/master/example_data/watchlist.csv?raw=True")
df.head()

,id_source,ra,dec
0,source_1,254.843418,1.646404
1,source_2,215.416280,13.229551


# Query objects within 10 deg of given positions, first detected in the last 10 days

In [5]:
from astropy.time import Time
nt = Time.now()
last_mjd_discovery = nt.mjd - 10

### Spatial crossmatch via ADQL cone search

A previous version of this notebook used a PostgreSQL-specific `q3c_join` plus a `WITH ... (VALUES ...)` common table expression (CTE) to crossmatch the watchlist against the database in a single query. Neither is part of ADQL, so on TAP we run one ADQL cone search per input position (`CONTAINS`/`CIRCLE`), letting the service do the spatial match server-side on its q3c index, and concatenate the results.

In [6]:
search_radius = 1 # [deg]

def build_query(row):
    return f"""
    SELECT
        '{row.id_source}' AS source_id, {row.ra} AS cat_ra, {row.dec} AS cat_dec,
        o.oid, o.meanra, o.meandec, o.firstmjd,
        DISTANCE(POINT('ICRS', o.meanra, o.meandec),
                 POINT('ICRS', {row.ra}, {row.dec})) AS dist
    FROM ztf.object AS o
    WHERE CONTAINS(POINT('ICRS', o.meanra, o.meandec),
                   CIRCLE('ICRS', {row.ra}, {row.dec}, {search_radius})) = 1
        AND o.firstmjd > {last_mjd_discovery}
    ORDER BY o.oid
    """

matches = cone_search_per_position(df, build_query)
print(matches.shape)
matches

(340, 8)


,source_id,cat_ra,cat_dec,oid,meanra,meandec,firstmjd,dist
0,source_1,254.843418,1.646404,ZTF19ablzrvd,255.532573,1.453183,61212.216991,0.715487
1,source_1,254.843418,1.646404,ZTF21aawgcgn,255.244744,2.061777,61216.264155,0.577433
2,source_1,254.843418,1.646404,ZTF21aawgenc,254.706323,1.269802,61213.361146,0.400764
3,source_1,254.843418,1.646404,ZTF26abcjigy,254.308100,0.913951,61208.289352,0.907141
4,source_1,254.843418,1.646404,ZTF26abcjigz,254.427248,0.872322,61208.289352,0.878814
...,...,...,...,...,...,...,...,...
335,source_2,215.416280,13.229551,ZTF26abdtqfy,214.826750,12.695495,61216.191042,0.784393
336,source_2,215.416280,13.229551,ZTF26abdttga,214.774347,13.843004,61216.213125,0.875112
337,source_2,215.416280,13.229551,ZTF26abdttkz,214.513394,12.789463,61216.213125,0.983649
338,source_2,215.416280,13.229551,ZTF26abdttlc,214.922206,13.318919,61216.213125,0.489108


# Query objects within 10 deg of given positions, first detected within the last 10 days, with SN probabilities > 0.4, and ranking = 1 in the stamp classifier

### Crossmatch + SN probability filter

Same per-position cone search as above, now joined to `ztf.probability` to keep only recent SN
candidates (stamp classifier, ranking 1, probability > 0.4). The join runs server-side on TAP,
replacing the old `q3c_join` + CTE query.

In [7]:
search_radius = 10 # [deg]

def build_query(row):
    return f"""
    SELECT
        '{row.id_source}' AS source_id, {row.ra} AS cat_ra, {row.dec} AS cat_dec,
        o.oid, o.meanra, o.meandec, o.firstmjd,
        DISTANCE(POINT('ICRS', o.meanra, o.meandec),
                 POINT('ICRS', {row.ra}, {row.dec})) AS dist,
        p.classifier_name, p.class_name, p.probability, p.ranking
    FROM ztf.object o
    INNER JOIN ztf.probability p ON p.oid = o.oid
    WHERE CONTAINS(POINT('ICRS', o.meanra, o.meandec),
                   CIRCLE('ICRS', {row.ra}, {row.dec}, {search_radius})) = 1
        AND o.firstmjd > {last_mjd_discovery}
        AND p.classifier_name = 'stamp_classifier'
        AND p.class_name = 'SN'
        AND p.ranking = 1
        AND p.probability > 0.4
    ORDER BY o.oid
    """

matches = cone_search_per_position(df, build_query)
print(matches.shape)
matches.head(100)

(5, 12)


,source_id,cat_ra,cat_dec,oid,meanra,meandec,firstmjd,dist,classifier_name,class_name,probability,ranking
0,source_1,254.843418,1.646404,ZTF26abdgnug,258.064070,4.829312,61213.366794,4.524127,stamp_classifier,SN,0.420403,1
1,source_1,254.843418,1.646404,ZTF26abdgotu,252.503465,-2.223545,61213.367268,4.522130,stamp_classifier,SN,0.441858,1
2,source_1,254.843418,1.646404,ZTF26abdujyh,256.103667,2.045999,61216.264155,1.321457,stamp_classifier,SN,0.408130,1
3,source_2,215.416280,13.229551,ZTF26abcikpj,221.658984,10.999759,61208.193773,6.497681,stamp_classifier,SN,0.496124,1
4,source_2,215.416280,13.229551,ZTF26abdukrq,210.769305,8.948051,61216.268831,6.254247,stamp_classifier,SN,0.453039,1


# Open the resulting objects in the ALeRCE explorer

In [8]:
url = "https://alerce.online/?" \
      + "&".join(["oid=%s" % oid for oid in matches.oid.values]) \
      + "&count=true&page=1&perPage=1000&sortDesc=false"
print(url)

https://alerce.online/?oid=ZTF26abdgnug&oid=ZTF26abdgotu&oid=ZTF26abdujyh&oid=ZTF26abcikpj&oid=ZTF26abdukrq&count=true&page=1&perPage=1000&sortDesc=false
